# Day 2: Tool Calling & Function Agents
### Agentic AI Hands-On Course | Dr. Kanthi Kiran Sirra | Sr. AI Engineer

**What you'll build today:**
- Understand how function calling works at the raw API level (JSON schema)
- Build custom tools with type hints, docstrings, and error handling
- Build a multi-tool agent that picks the right tool automatically
- Chain multiple tools together for complex multi-step reasoning
- Understand the ReAct loop from the inside

**Framework:** Smolagents (HuggingFace) — same as Day 1
**LLM:** Groq (Llama 3.3 70B) — via LiteLLM

## 0. Setup — Load your API keys

Run this cell first. You should see green checkmarks for all API keys.

In [ ]:
# === COLAB USERS: Run this cell first to install packages ===
# !pip install "smolagents[litellm]" ddgs python-dotenv -q

In [ ]:
# === COLAB USERS: Uncomment the block below ===
# from google.colab import userdata
# import os
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
# os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# === LOCAL USERS: Just run this cell ===
import os
from dotenv import load_dotenv
load_dotenv()

# Verify keys
for key in ["GROQ_API_KEY", "GOOGLE_API_KEY", "HF_TOKEN"]:
    val = os.getenv(key)
    status = "✅" if val else "❌ MISSING"
    print(f"{key}: {status}")

---
## 1. 🎯 Hook — Watch a 5-tool agent solve a complex problem

Before we learn HOW tools work, let's see WHAT they can do.

This agent has 5 tools and will decide which ones to use — **on its own**.

In [ ]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, LiteLLMModel, tool
import os
import litellm
litellm.suppress_debug_info = True

# --- Define 5 tools ---

@tool
def calculate(expression: str) -> str:
    """Evaluates a mathematical expression and returns the result.
    
    Args:
        expression: A mathematical expression like '2 + 2' or '(15 * 3) / 4.5'
    """
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {e}"

@tool
def get_word_count(text: str) -> str:
    """Counts the number of words, characters, and sentences in the given text.
    
    Args:
        text: The text to analyze
    """
    words = len(text.split())
    chars = len(text)
    sentences = text.count('.') + text.count('!') + text.count('?')
    return f"Words: {words}, Characters: {chars}, Sentences: {sentences}"

@tool
def convert_temperature(value: float, from_unit: str, to_unit: str) -> str:
    """Converts temperature between Celsius, Fahrenheit, and Kelvin.
    
    Args:
        value: The temperature value to convert
        from_unit: Source unit - 'C', 'F', or 'K'
        to_unit: Target unit - 'C', 'F', or 'K'
    """
    value = float(value)  # tool decorator may pass strings
    # First convert to Celsius
    if from_unit == 'F':
        celsius = (value - 32) * 5/9
    elif from_unit == 'K':
        celsius = value - 273.15
    else:
        celsius = value
    
    # Then convert to target
    if to_unit == 'F':
        result = celsius * 9/5 + 32
    elif to_unit == 'K':
        result = celsius + 273.15
    else:
        result = celsius
    
    return f"{value}°{from_unit} = {result:.2f}°{to_unit}"

@tool
def reverse_text(text: str) -> str:
    """Reverses the given text and also checks if it's a palindrome.
    
    Args:
        text: The text to reverse
    """
    reversed_text = text[::-1]
    is_palindrome = text.lower().replace(" ", "") == reversed_text.lower().replace(" ", "")
    result = f"Reversed: {reversed_text}"
    if is_palindrome:
        result += " (This is a palindrome!)"
    return result

# --- Create the agent ---
groq_model = LiteLLMModel(
    model_id="groq/llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY")
)

agent = CodeAgent(
    tools=[DuckDuckGoSearchTool(), calculate, get_word_count, convert_temperature, reverse_text],
    model=groq_model
)

# --- Ask it something that needs calculation ---
print("--- Query: Needs calculator ---")
result = agent.run("What is 1947 multiplied by 77, then add 2025?")
print(result)

**Watch what happened:** The agent had 5 tools available, but it chose `calculate` because the question was about math. It didn't use web search, word count, or temperature conversion — it **reasoned** about which tool fits.

Now let's try a different question with the SAME agent:

In [ ]:
# Same agent, different question — should pick a DIFFERENT tool
print("--- Query: Needs web search ---")
result = agent.run("What is the current population of Visakhapatnam?")
print(result)

**Different question → Different tool → Automatically.** This is the core of what makes an agent an AGENT.

Now the question is: **HOW does this work under the hood?** That's what today is about.

---
## 2. What is Function Calling? — The Raw API Level

When you use `@tool` and `CodeAgent`, Smolagents hides a lot of complexity. Let's peel back the layers.

**Function calling** is a feature where the LLM doesn't just generate text — it generates a **structured JSON object** that says "call this function with these arguments."

Let's see how this works at the raw API level with Groq:

In [ ]:
import requests
import json

# Define tools as JSON schemas — this is what the LLM actually sees
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluates a mathematical expression and returns the result",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "A mathematical expression like '2 + 2' or '15 * 3'"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Gets the current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The city name, e.g. 'Hyderabad'"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

# Send the request with tools
url = "https://api.groq.com/openai/v1/chat/completions"
headers = {
    "Authorization": f"Bearer {os.getenv('GROQ_API_KEY')}",
    "Content-Type": "application/json"
}

payload = {
    "model": "llama-3.3-70b-versatile",
    "messages": [
        {"role": "user", "content": "What is 245 multiplied by 38?"}
    ],
    "tools": tools_schema,  # <-- THIS IS THE KEY ADDITION
    "tool_choice": "auto"   # Let the LLM decide which tool (or none)
}

response = requests.post(url, headers=headers, json=payload)
data = response.json()

# What did the LLM respond with?
message = data["choices"][0]["message"]
print("=== LLM Response ===")
print(json.dumps(message, indent=2))

### What just happened?

Instead of answering "245 * 38 = 9310", the LLM responded with a **tool call**:

```json
{
  "tool_calls": [{
    "function": {
      "name": "calculate",
      "arguments": "{"expression": "245 * 38"}"
    }
  }]
}
```

The LLM is saying: *"I don't want to calculate this myself. Please call the `calculate` function with the expression `245 * 38`, and give me the result."*

**This is the fundamental mechanism behind ALL agent frameworks:**
1. You tell the LLM what tools exist (JSON schemas)
2. The LLM decides if/which tool to call
3. The LLM generates structured JSON (not free text)
4. YOUR code executes the function and returns the result
5. The LLM uses the result to generate the final answer

Let's complete the loop — execute the tool call and feed the result back:

In [ ]:
# Step 1: Extract the tool call from the LLM response
tool_call = message["tool_calls"][0]
function_name = tool_call["function"]["name"]
arguments = json.loads(tool_call["function"]["arguments"])

print(f"LLM wants to call: {function_name}")
print(f"With arguments: {arguments}")
print()

# Step 2: Execute the function ourselves
if function_name == "calculate":
    result = str(eval(arguments["expression"]))
elif function_name == "get_weather":
    result = f"Weather in {arguments['city']}: 32°C, Sunny"  # Fake for demo
else:
    result = "Unknown function"

print(f"Function result: {result}")
print()

# Step 3: Send the result BACK to the LLM
payload2 = {
    "model": "llama-3.3-70b-versatile",
    "messages": [
        {"role": "user", "content": "What is 245 multiplied by 38?"},
        message,  # The LLM's tool call response
        {
            "role": "tool",
            "tool_call_id": tool_call["id"],
            "content": result  # The function's result
        }
    ],
    "tools": tools_schema
}

response2 = requests.post(url, headers=headers, json=payload2)
data2 = response2.json()

final_answer = data2["choices"][0]["message"]["content"]
print(f"=== Final Answer ===")
print(final_answer)

### The Complete Function Calling Loop

```
You: "What is 245 × 38?"
       ↓
LLM: "I want to call calculate(expression='245 * 38')"     ← tool_calls in response
       ↓
Your Code: eval("245 * 38") → 9310                          ← YOU execute it
       ↓
You → LLM: "The result of calculate was: 9310"              ← role: "tool"
       ↓
LLM: "245 multiplied by 38 equals 9,310."                   ← Final answer
```

**That's 2 API calls for one question.** This is why agents eat tokens fast.

**Key insight:** The LLM never executes code. It only GENERATES the function call as JSON. YOUR code does the actual execution. The LLM is the brain, your code is the hands.

### Now let's try — what if the question doesn't need any tool?

In [ ]:
# A question that doesn't need tools
payload3 = {
    "model": "llama-3.3-70b-versatile",
    "messages": [
        {"role": "user", "content": "What is the capital of India?"}
    ],
    "tools": tools_schema,
    "tool_choice": "auto"
}

response3 = requests.post(url, headers=headers, json=payload3)
data3 = response3.json()
message3 = data3["choices"][0]["message"]

print("=== Response for 'Capital of India' ===")
if message3.get("tool_calls"):
    print(f"Tool called: {message3['tool_calls'][0]['function']['name']}")
else:
    print(f"Direct answer (no tool): {message3['content']}")

**The LLM decided it doesn't need any tool** — it already knows the capital of India from training data. This is the intelligence of function calling: the LLM only calls tools when it **needs** them.

---

## 3. The @tool Decorator — Deep Dive

Now you understand what function calling is at the raw level. Smolagents' `@tool` decorator **automates all of that** — it reads your function and generates the JSON schema automatically.

Let's build tools with increasing complexity:

### 3.1 Anatomy of a good tool

A good tool has 4 parts: decorator, type hints, docstring, and error handling.

In [ ]:
# ANATOMY OF A GOOD TOOL

@tool
def unit_converter(value: float, from_unit: str, to_unit: str) -> str:
    """Converts between common units of measurement.
    
    Supports: km/miles, kg/pounds, liters/gallons, cm/inches
    
    Args:
        value: The numeric value to convert
        from_unit: Source unit (km, miles, kg, pounds, liters, gallons, cm, inches)
        to_unit: Target unit (km, miles, kg, pounds, liters, gallons, cm, inches)
    """
    # Conversion factors to base units
    conversions = {
        'km': ('distance', 1.0),
        'miles': ('distance', 1.60934),
        'kg': ('weight', 1.0),
        'pounds': ('weight', 0.453592),
        'liters': ('volume', 1.0),
        'gallons': ('volume', 3.78541),
        'cm': ('length', 1.0),
        'inches': ('length', 2.54),
    }
    
    if from_unit not in conversions or to_unit not in conversions:
        return f"Error: Unknown unit. Supported: {', '.join(conversions.keys())}"
    
    from_type, from_factor = conversions[from_unit]
    to_type, to_factor = conversions[to_unit]
    
    if from_type != to_type:
        return f"Error: Cannot convert {from_unit} ({from_type}) to {to_unit} ({to_type})"
    
    # Convert: source → base → target
    value = float(value)  # tool decorator may pass strings
    base_value = value * from_factor
    result = base_value / to_factor
    
    return f"{value} {from_unit} = {result:.4f} {to_unit}"

# Test it manually first
print(unit_converter(10, "km", "miles"))
print(unit_converter(100, "kg", "pounds"))
print(unit_converter(5, "gallons", "liters"))

### What makes this a GOOD tool?

1. **Clear docstring** — the agent reads "Converts between common units" and knows when to use it
2. **Specific Args** — the agent knows it needs `value`, `from_unit`, `to_unit`
3. **Supported units listed** — the agent can read the supported options
4. **Error handling** — returns helpful error messages instead of crashing
5. **Type hints** — `float`, `str` → `str` tells the agent the input/output types

### 3.2 Tools with external data

In [ ]:
import datetime

@tool
def get_date_info(date_string: str) -> str:
    """Gets information about a specific date — day of week, days until/since, and fun facts.
    
    Args:
        date_string: A date in YYYY-MM-DD format, e.g. '2025-01-26'
    """
    try:
        date = datetime.datetime.strptime(date_string, "%Y-%m-%d")
        today = datetime.datetime.now()
        
        day_name = date.strftime("%A")
        diff = (date - today).days
        
        if diff > 0:
            time_info = f"{diff} days from now"
        elif diff < 0:
            time_info = f"{abs(diff)} days ago"
        else:
            time_info = "Today!"
        
        # Day of year
        day_of_year = date.timetuple().tm_yday
        
        return f"Date: {date.strftime('%B %d, %Y')} ({day_name}) | {time_info} | Day {day_of_year} of the year"
    except ValueError:
        return f"Error: Invalid date format. Please use YYYY-MM-DD (e.g., '2025-01-26')"

# Test
print(get_date_info("2025-08-15"))
print(get_date_info("2000-01-01"))

### 3.3 Tools that process text

In [ ]:
@tool
def text_analyzer(text: str) -> str:
    """Analyzes text and returns detailed statistics including readability metrics.
    
    Args:
        text: The text to analyze
    """
    words = text.split()
    word_count = len(words)
    char_count = len(text)
    sentence_count = text.count('.') + text.count('!') + text.count('?') or 1
    
    avg_word_length = sum(len(w) for w in words) / max(word_count, 1)
    avg_sentence_length = word_count / max(sentence_count, 1)
    
    # Simple readability (Flesch-like approximation)
    if avg_sentence_length < 10:
        readability = "Very Easy"
    elif avg_sentence_length < 15:
        readability = "Easy"
    elif avg_sentence_length < 20:
        readability = "Moderate"
    else:
        readability = "Complex"
    
    # Find longest and shortest words
    longest = max(words, key=len) if words else ""
    shortest = min((w for w in words if len(w) > 1), key=len, default="")
    
    return (f"Words: {word_count} | Characters: {char_count} | Sentences: {sentence_count}\n"
            f"Avg word length: {avg_word_length:.1f} chars | Avg sentence: {avg_sentence_length:.1f} words\n"
            f"Readability: {readability}\n"
            f"Longest word: '{longest}' ({len(longest)} chars) | Shortest: '{shortest}' ({len(shortest)} chars)")

# Test
print(text_analyzer("Artificial Intelligence is transforming how we build software. Agents can now reason and use tools. This changes everything."))

---
## 4. Multi-Tool Agent — The Agent Picks the Right Tool

Now let's give the agent ALL our tools and see how it decides which one to use for different questions:

In [ ]:
# Create a super-agent with ALL tools
super_agent = CodeAgent(
    tools=[
        DuckDuckGoSearchTool(),  # Pre-built: web search
        calculate,                # Math
        unit_converter,           # Unit conversion
        convert_temperature,      # Temperature
        get_date_info,            # Date info
        text_analyzer,            # Text analysis
        reverse_text,             # Reverse + palindrome
    ],
    model=groq_model
)

print(f"Agent has {len(super_agent.tools)} tools ready!")
print("Tools:", list(super_agent.tools.keys()) if isinstance(super_agent.tools, dict) else [t.name for t in super_agent.tools])

Now let's test it with different types of questions — watch which tool it picks:

In [ ]:
# Query 1: Should use calculate
print("=" * 50)
print("Query 1: Math question")
print("=" * 50)
result = super_agent.run("If I buy 15 items at 499 rupees each and get a 12% discount, what do I pay?")
print(result)

In [ ]:
# Query 2: Should use unit_converter
print("=" * 50)
print("Query 2: Unit conversion")
print("=" * 50)
result = super_agent.run("I ran 5 miles today. How many kilometers is that?")
print(result)

In [ ]:
# Query 3: Should use get_date_info
print("=" * 50)
print("Query 3: Date question")
print("=" * 50)
result = super_agent.run("What day of the week was India's independence day in 1947? (August 15, 1947)")
print(result)

In [ ]:
# Query 4: Should use text_analyzer
print("=" * 50)
print("Query 4: Text analysis")
print("=" * 50)
result = super_agent.run("Analyze this text: Machine learning is a subset of artificial intelligence that enables systems to learn from data without being explicitly programmed.")
print(result)

### 🧠 What just happened?

Same agent, same code, 4 different questions → 4 different tools chosen automatically.

The LLM reads each question, looks at ALL 7 tool descriptions, and picks the most relevant one. This is **tool selection** — the core skill of an AI agent.

---

## 5. Tool Chaining — Agent Uses Multiple Tools for One Question

The real power of agents is when they use **multiple tools in sequence** to answer ONE question.

Let's ask questions that need more than one tool:

In [ ]:
# This question needs: calculate + unit_converter (or just calculate)
print("=" * 50)
print("Query: Multi-step — needs chaining!")
print("=" * 50)
result = super_agent.run(
    "I drove 120 km to work and 120 km back. "
    "My car uses 8 liters of fuel per 100 km. "
    "If fuel costs 105 rupees per liter, how much did I spend on fuel today?"
)
print(result)

**Watch the agent's reasoning:**
1. Total distance = 240 km
2. Fuel used = 240 * 8/100 = 19.2 liters
3. Cost = 19.2 * 105 = 2016 rupees

The agent might use the calculate tool **multiple times** in one run — that's tool chaining!

In [ ]:
# Another chaining example — needs search + reasoning
print("=" * 50)
print("Query: Needs web search + reasoning")
print("=" * 50)
result = super_agent.run(
    "How many days are there until the next Republic Day of India (January 26)?"
)
print(result)

---
## 6. Error Handling — What Happens When Tools Fail?

Good tools handle errors gracefully. Let's see what happens with bad inputs:

In [ ]:
# Test error handling in our tools
print("=== Error handling tests ===")
print()

# Bad math expression
print("Bad math:", calculate("hello + world"))
print()

# Bad unit conversion
print("Bad units:", unit_converter("10", "km", "pounds"))
print()

# Bad date
print("Bad date:", get_date_info("not-a-date"))

**Why error handling matters for agents:**

If a tool crashes with an exception, the entire agent run fails. But if it returns a helpful error message, the agent can:
1. Read the error
2. Try a different approach
3. Or report the issue to the user

**Rule of thumb:** Tools should NEVER raise exceptions. Always return error strings.

---
## 7. 🎯 Your Turn — Build 3 Custom Tools + Agent

**Challenge:** Create 3 NEW tools that solve real problems. Then build an agent with all your tools.

**Ideas for tools:**
- `bmi_calculator(weight_kg, height_cm)` — calculates BMI and category
- `password_strength(password)` — checks password strength
- `emi_calculator(principal, rate, months)` — loan EMI calculation
- `gpa_calculator(marks_list)` — CGPA from marks
- `time_zone_converter(time, from_zone, to_zone)` — time zone conversion
- `text_to_hashtags(text)` — generates hashtags from text
- `recipe_scaler(recipe, servings)` — scale recipe ingredients

**Requirements:**
1. Each tool must have `@tool` decorator
2. Each tool must have a clear docstring with Args
3. Each tool must have type hints
4. Each tool must handle errors gracefully (no crashes!)

In [ ]:
# === YOUR TOOL 1 ===
@tool
def bmi_calculator(weight_kg: float, height_cm: float) -> str:
    """Calculates Body Mass Index (BMI) and returns the category.
    
    Args:
        weight_kg: Weight in kilograms
        height_cm: Height in centimeters
    """
    # TODO: Implement this!
    weight_kg = float(weight_kg)  # tool decorator passes strings!
    height_cm = float(height_cm)
    # Formula: BMI = weight / (height_in_meters ** 2)
    # Categories: <18.5 Underweight, 18.5-24.9 Normal, 25-29.9 Overweight, 30+ Obese
    pass

# === YOUR TOOL 2 ===
# TODO: Create your second tool

# === YOUR TOOL 3 ===
# TODO: Create your third tool

# === CREATE YOUR AGENT ===
# my_agent = CodeAgent(
#     tools=[bmi_calculator, your_tool_2, your_tool_3],
#     model=groq_model
# )

# === TEST YOUR AGENT ===
# result = my_agent.run("Calculate my BMI. I weigh 70 kg and I'm 175 cm tall.")
# print(result)

---
## 8. Recap — What You Built Today

### Concepts:
- **Function calling** = LLM generates structured JSON to call functions (not free text)
- **JSON schema** = how you describe tools to the LLM (name, description, parameters)
- **Tool selection** = LLM reads all tool descriptions and picks the best one
- **Tool chaining** = agent uses multiple tools in sequence for complex questions
- **Error handling** = tools should return error strings, never crash

### The Function Calling Loop:
```
User question → LLM picks a tool → Your code executes → Result back to LLM → Final answer
```

### What `@tool` automates:
```
Raw API: Define JSON schema + parse tool_calls + execute + feed back + repeat
@tool:   Just write a Python function with a docstring → Smolagents handles the rest
```

### Key Takeaways:
1. The LLM **never executes code** — it only generates the function call as JSON
2. YOUR code does the actual execution — the LLM is the brain, your code is the hands
3. Good tools have: clear docstrings, type hints, error handling
4. The docstring is CRITICAL — it's how the agent decides when to use your tool
5. Agents eat 2-5x more tokens than simple chat because of the multi-turn tool calling loop

---

### Tomorrow: Day 3 — Agent Memory Systems
How do we make agents remember previous conversations? Short-term memory, long-term memory, and the difference between a forgetful chatbot and a memory-enabled agent.